# Homework 4: Optional Final Project (A+ Grade Bump)

- This homework template guides you through presenting your final project analysis.
- Use this notebook to:
    - Generate all visualizations/results and report findings with a pipeline then simply view the results here
    - Generate figures/analysis with imported scripts to produce visualizations/results, and report findings.

> **Note for Beginners:** Running modular Python scripts from inside a Jupyter notebook can sometimes lead to import path or dependency issues if the working directory changes. If you encounter import errors, make sure you add the path of your script folder to `sys.path`, or execute your pipeline directly from your terminal using:
```bash
uv run python src/final_project/ryu-hasegawa/basic/pipeline.py
```

## A. Describe Project

### **Guidance**
- State the policy question, puzzle, or social science problem you are addressing.
- Frame your central hypothesis and the expected relationship between your variables.
- Describe the scope of your analysis (e.g., geographical regions, years covered).
- Highlight the datasets you selected to examine this question.

### **Project Summary**
- **Project Title:** Is China's Overseas Lending Procyclical? A Panel Analysis of Borrower vs. Lender Business Cycles
- **Student Name:** [Your Name]
- **Policy Relevance Statement:** As China has become the largest bilateral official creditor to low- and middle-income countries, understanding whether its lending stabilizes or destabilizes borrower economies over the business cycle matters directly for debt sustainability analysis and for alternative official lenders (e.g., JBIC, NEXI, JICA) considering how to position themselves as counter-cyclical financing sources.
- **Central Hypothesis:** China's bilateral official lending is (H1) acyclical with respect to the borrower country's own business cycle, but (H2) procyclical with respect to China's own business cycle, and (H3) this procyclicality has strengthened since the 2008 Global Financial Crisis. This design follows Avellan, Galindo, Gomez & Lotti (2024, *Emerging Markets Review*), "The Cyclicality of Official Bilateral Lending."

---

## 1. Download Data

### **Guidance**
- Run the data acquisition step by executing or importing from your `data.py` file.
- Ensure your script programmatically downloads the datasets and saves them locally.
- Verify the download by displaying the destination folder structure and confirming files are saved.

### **Data Acquisition Details**
- **Primary Data Source:** World Bank International Debt Statistics (IDS), source 6, with the `counterpart-area` dimension filtered to China (code 730), indicator `DT.DOD.BLAT.CD` (PPG bilateral debt stock, current US$)
- **Secondary Data Source:** IMF DataMapper API (World Economic Outlook), indicator `NGDP_RPCH` (real GDP growth, annual %)
- **Variables Retrieved:** `country`, `year`, `gdp_growth` (borrower and China), `china_debt_usd` (PPG bilateral debt stock owed to China)
- **Local Storage Path:** `data/final_project/ryu-hasegawa/weo_growth_raw.csv`, `data/final_project/ryu-hasegawa/china_debt_raw.csv`


In [ ]:
# Import and run your data acquisition script
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../../src/final_project/ryu-hasegawa/basic').resolve()))
import data
growth_df, debt_df = data.run()
growth_df.head()


## 2. Manipulate Data

### **Guidance**
- Preprocess, clean, and merge your raw files by running your `manipulate.py` file.
- Filter out missing values, fix data types, handle outliers, and align column names.
- Merge your datasets using standardized index columns (e.g., country name, date, country-year).
- Verify the final shape of the combined dataset and display a preview (`.head()`).

### **Preprocessing Summary**
- **Merging Strategy:** Country-year inner merge of borrower GDP growth with China-owed bilateral debt stock; China's own GDP growth merged on year only (constant across borrowers within a year)
- **Derived Variables:** `china_debt_change` (YoY change in debt stock, proxy for new net lending flow), `post_gfc` (year >= 2009 dummy), `china_growth_x_post_gfc` (interaction term for H3)
- **Clean Data Path:** `data/final_project/ryu-hasegawa/processed_china_lending_panel.csv`


In [ ]:
import manipulate
df_clean = manipulate.run()
df_clean.head()


## 3. Visualize Data

### **Guidance**
- Generate publication-quality visualizations by calling functions in your `graph.py` file.
- Your graphs must include proper axis labels, descriptive titles, accessible colors, and clear legends.
- Display the figures directly in this section by importing your graphing functions.

### **Visualizations & Observations**
- **Figure 1 Key Takeaway:** The scatter of borrower real GDP growth against the change in China-owed PPG bilateral debt shows a nearly flat regression line across the 12 borrower countries, with points scattered widely at almost every level of borrower growth. This is visually consistent with H1 (acyclicality with respect to the borrower's own business cycle): borrowers do not appear to receive systematically more or less new lending simply because they are in a boom or a downturn. One large outlier (Pakistan) pulls the slope slightly positive, which is addressed in the model interpretation below.
- **Figure 2 Key Takeaway:** China's own real GDP growth trended downward from double digits in the 1980s-2000s to roughly 6-8% after 2010, while aggregate new lending to the 12 borrower countries expanded sharply starting around 2011-2013 (coinciding with the Belt and Road Initiative) before falling off after 2018-2020. The raw time series alone do not show a simple positive year-to-year co-movement (lending rose while growth was falling), which is why the regression's post-2008 interaction term, rather than this chart alone, is the more reliable test of H2.


In [ ]:
import graph
graph.run()


## 4. Model Data

### **Guidance**
- Run statistical modeling (such as OLS regression) using functions in your `model.py` file.
- Specify your dependent variable and independent variables clearly.
- Print out the full model summary (e.g., coefficient table, standard errors, R-squared, p-values).
- Summarize your model findings and state whether they support your initial hypothesis.

### **Model Specifications & Interpretation**
- **Model Type:** Panel Ordinary Least Squares (PanelOLS) with Entity Fixed Effects
- **Dependent Variable:** `china_debt_change` (YoY change in PPG bilateral debt owed to China, proxy for new lending flow)
- **Independent Variables:** `borrower_gdp_growth`, `china_gdp_growth`, `china_growth_x_post_gfc`
- **Key Conclusion:** The PanelOLS regression (R-squared = 0.1248, N = 440, 12 entities) shows that `borrower_gdp_growth` is not statistically significant (coef = 7.98e6, p = 0.183), supporting **H1**: China's bilateral lending does not respond systematically to the borrower's own business cycle. `china_gdp_growth` alone is also not significant (coef = -9.53e6, p = 0.271), so **H2 is not supported in its simple form** for the pre-2008 period. However, the interaction term `china_growth_x_post_gfc` is large, positive, and highly significant (coef = 4.57e7, p < 0.001), meaning the relationship between China's own growth and its lending flipped sign and became strongly positive after the 2008 Global Financial Crisis (post-2008 marginal effect ≈ -9.53e6 + 4.57e7 ≈ +3.6e7). This supports **H3**: procyclicality with respect to China's own business cycle emerged specifically in the post-GFC period, consistent with the direction (though not the exact magnitude) of Avellan, Galindo, Gomez & Lotti (2024)'s finding that Chinese official lending became more procyclical to the lender's own cycle after 2008. One caveat: Pakistan is a high-leverage outlier in the borrower-growth relationship (Figure 1), so a robustness check excluding Pakistan would strengthen this conclusion further.


In [ ]:
import model
summary = model.run()
